In [ ]:
import sys, os, warnings, subprocess
subprocess.run(["pip", "install", ".", "--quiet"], check=True,
               cwd="/mnt/storage_ssd_raid/research_projects/intrinsic_neural_timescales")
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

import numpy as np
import scipy as sp
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec

%matplotlib inline
plt.rcParams.update({
    'font.size':         8,
    'figure.dpi':        300,
    'svg.fonttype':      'none',
    'font.family':       'sans-serif',
    'font.sans-serif':   ['Roboto'],
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

sys.path.extend([r'/mnt/storage_ssd_raid/research_projects/snaplab_tools'])

from src.notebook import load_notebook_data
from src.timescales import (create_ints_dataframe_custom_grouping, convert_data_dict,
                             average_data_dict_converted, compute_deflections)
from src.plotting import create_sydnor_sa_colormap
from src.gams import (fit_all_panels, apply_region_mask, get_series_colors, get_draw_order,
                      style_ax, compute_deriv_sa_corr, bootstrap_gam_maps,
                      add_sa_decile_colorbar, plot_deriv_sa_corr_with_ci, compute_partial_r2_age)

# Load data

In [ ]:
dataset       = 'HCP-D' # 'HCP-D' 'PNC'
which_modality = 'FMRI'
n_regions = 400

if dataset == 'HCP-D':
    # output_type   = 'timescale_auc_zero_crossing'
    output_type   = 'timescale_auc_zero_crossing_bp-0.01-0.08hz'
elif dataset == 'PNC':
    output_type   = 'timescale_auc_zero_crossing'
    # output_type   = 'timescale_zero_crossing'
    # output_type   = 'timescale_zero_crossing_robust_nc2'
    # output_type   = 'timescale_zero_crossing_robust_nc3'

notebook_data = load_notebook_data(dataset=dataset, which_modality=which_modality,
                                   output_type=output_type, use_resids=True,
                                   load_subject_data=False, n_regions=n_regions)

In [ ]:
timescales_processor = notebook_data['timescales_processor']
data_dict            = notebook_data['data_dict']
results_dir          = notebook_data['results_dir']
df_neuro             = notebook_data['supporting_data']['df_neuro']

sa_axis = df_neuro['SA_Axis'].values

nas_scratch_dir = f'/mnt/lindenmp_nas/lindenmp/work/figure_scratch/intrinsic_neural_timescales/{dataset}'
os.makedirs(nas_scratch_dir, exist_ok=True)

In [ ]:
sa_axis.shape

# Prepare data

In [ ]:
timescales_processor.match_registry_to_behavior()
df_behavior = timescales_processor.df_behavior_matched.copy()

if dataset == 'HCP-D':
    rest_scan = 'task-Rest'
    # task_scan = 'tfMRIEMOTIONPA'
    task_scan = 'tfMRICARITPA'
    # task_scan = 'tfMRIGUESSINGPA'
elif dataset == 'PNC':
    rest_scan = 'rest'
    task_scan = 'frac2back'

data_dict_tsnr      = notebook_data['data_dict_tsnr']
data_dict_converted = convert_data_dict(data_dict, timescales_processor.scan_registry)
data_dict_centiles  = convert_data_dict(data_dict, timescales_processor.scan_registry,
                                        brain_map=sa_axis)
data_dict_converted = average_data_dict_converted(data_dict_converted, average_all_rest=True)
data_dict_centiles  = average_data_dict_converted(data_dict_centiles,  average_all_rest=True)
df_tsnr             = create_ints_dataframe_custom_grouping(data_dict_tsnr, average_all_rest=True,
                                                            n_regions=n_regions)

# Fit GAMs

In [ ]:
df_behavior

In [ ]:
# ── GAM + bootstrap configuration ────────────────────────────────
n_splines           = 4
lam                 = 0.1
absolute            = True
n_boot              = 5000      # bootstrap iterations (derivative × S-A CI + fig5 stage maps)
boot_seed           = 42        # resampling seed (shared by both bootstrap reductions)
inflection_age      = 17        # stage-1/stage-2 boundary (yr) for the exported bootstrap stage maps
N_JOBS              = 64        # joblib workers for the bootstrap (-1 = all cores)

# The GAM fits + bootstrap are slow, so their outputs are cached to gam_cache_file and reloaded on
# re-runs. Set FORCE_RERUN = True to recompute from scratch and overwrite the cache.
FORCE_RERUN    = False
gam_out_dir    = os.path.join(timescales_processor.output_dir, 'results', dataset, 'gam')
os.makedirs(gam_out_dir, exist_ok=True)
gam_cache_file = os.path.join(gam_out_dir, f'{dataset}_{n_regions}_{task_scan}_regional_gam_results.npy')

In [ ]:
# Load the cached GAM + bootstrap outputs if present, else compute them from scratch and save.
if (not FORCE_RERUN) and os.path.exists(gam_cache_file):
    save_dict = np.load(gam_cache_file, allow_pickle=True).item()
    print(f'Loaded cached GAM + bootstrap outputs <- {gam_cache_file}')
    print('  (set FORCE_RERUN = True in the config cell above to recompute and overwrite)')
else:
    # ===== Region- and decile-level data =====
    rest_full = data_dict_converted[rest_scan]
    try:
        task_full = data_dict_converted[task_scan]
    except (KeyError, TypeError):
        task_full = data_dict_converted[rest_scan]
    n_subs = task_full.shape[0]
    n_regions = task_full.shape[1]

    # Covariate data (age smooth + nuisance: sex, meanFD, and each subject's mean rest INT)
    age = df_behavior['age'].values
    if dataset == 'HCP-D':
        nuisance = np.concat((df_behavior[['sex', 'meanFD']].values,
                              np.nanmean(rest_full, axis=1)[:, np.newaxis]), axis=1)
    elif dataset == 'PNC':
        nuisance = np.concat((df_behavior[['sex', 'meanFD_rest']].values,
                              np.nanmean(rest_full, axis=1)[:, np.newaxis]), axis=1)
    else:
        nuisance = None
    age_grid = np.linspace(age.min(), age.max(), 200)

    deflection_full = compute_deflections(rest_full, task_full, absolute=absolute)

    rest_dec = data_dict_centiles[rest_scan]
    try:
        task_dec = data_dict_centiles[task_scan]
    except (KeyError, TypeError):
        task_dec = data_dict_centiles[rest_scan]
    deflection_dec = compute_deflections(rest_dec, task_dec, absolute=absolute)
    n_deciles = rest_dec.shape[1]

    # ===== Fit GAMs (decile + region) =====
    results_dec = fit_all_panels(age, rest_dec, task_dec, deflection_dec, age_grid,
                                 n_splines=n_splines, lam=lam, demean=True, show_ci=False,
                                 nuisance=nuisance)

    apply_tsnr_mask = False  # set True to exclude low-tSNR regions
    if apply_tsnr_mask:
        region_mask = (df_tsnr[task_scan] < np.quantile(df_tsnr[task_scan], q=0.1)).values
    else:
        region_mask = None
    rest_reg, task_reg, defl_reg, sa_axis_reg = apply_region_mask(
        region_mask, rest_full, task_full, deflection_full, sa_axis=sa_axis)
    results_reg = fit_all_panels(age, rest_reg, task_reg, defl_reg, age_grid,
                                 n_splines=n_splines, lam=lam, demean=True, show_ci=False,
                                 nuisance=nuisance)
    print(f'GAMs fit. Decile {deflection_dec.shape} | Regional {defl_reg.shape} | '
          f'Nuisance {None if nuisance is None else nuisance.shape}')

    # ===== Bootstrap: each subject-resample refit is reduced ONCE to both the derivative × S-A
    # axis correlation curve (this figure's CI) AND the stage-1/stage-2 net-change maps (exported
    # for Figure 5's Yeo-7 enrichment, so fig5 does zero GAM fitting). Parallel over resamples. =====
    boot = bootstrap_gam_maps(
        defl_reg, age, age_grid, sa_axis_vals=sa_axis_reg, inflection_age=inflection_age,
        n_splines=n_splines, lam=lam, demean=True, nuisance=nuisance, annualized=True,
        n_boot=n_boot, method='pearson', random_state=boot_seed, n_jobs=N_JOBS)
    boot_curves = boot['sa_curves']
    boot_stage1, boot_stage2 = boot['stage1'], boot['stage2']
    r_ci_lo = np.nanpercentile(boot_curves, 2.5,  axis=0)
    r_ci_hi = np.nanpercentile(boot_curves, 97.5, axis=0)
    print(f'Bootstrap done ({n_boot} iters, n_jobs={N_JOBS}). 95% CI at age midpoint: '
          f'[{r_ci_lo[100]:.3f}, {r_ci_hi[100]:.3f}]')
    print(f'Bootstrap stage maps for fig5: stage1 {boot_stage1.shape}, stage2 {boot_stage2.shape} '
          f'(inflection {inflection_age} yr).')

    # ===== Assemble outputs (figure inputs + cross-notebook export for fig5) =====
    save_dict = {
        'age_grid':    age_grid,
        'sa_axis_reg': sa_axis_reg,
        'task_scan':   task_scan,
        'n_deciles':   n_deciles,
    }
    for sig in ('rest', 'task', 'deflection'):
        save_dict[f'{sig}_deriv']     = results_reg[sig]['deriv']
        save_dict[f'{sig}_pred']      = results_reg[sig]['pred']
        save_dict[f'{sig}_deriv_dec'] = results_dec[sig]['deriv']
        save_dict[f'{sig}_pred_dec']  = results_dec[sig]['pred']
        save_dict[f'{sig}_partial_r2'] = compute_partial_r2_age(
            {'rest': rest_reg, 'task': task_reg, 'deflection': defl_reg}[sig],
            age, results_reg[sig]['gams'], nuisance=nuisance, demean=True
        )
    save_dict['r_vals']      = compute_deriv_sa_corr(
        results_reg['deflection']['deriv'], sa_axis_reg, method='pearson')
    save_dict['boot_curves']    = boot_curves
    save_dict['boot_stage1']    = boot_stage1     # (n_boot, n_regions) fig5 enrichment bootstrap
    save_dict['boot_stage2']    = boot_stage2
    save_dict['inflection_age'] = inflection_age
    save_dict['n_boot']         = n_boot
    save_dict['boot_seed']      = boot_seed
    save_dict['r_ci_lo']     = r_ci_lo
    save_dict['r_ci_hi']     = r_ci_hi
    save_dict['rest_reg']    = rest_reg
    save_dict['defl_reg']    = defl_reg
    save_dict['sa_axis']     = sa_axis
    save_dict['df_behavior'] = df_behavior

    np.save(gam_cache_file, save_dict)
    print(f'Computed and saved GAM + bootstrap outputs -> {gam_cache_file}')
    for sig in ('rest', 'task', 'deflection'):
        r2 = save_dict[f'{sig}_partial_r2']
        print(f"  {sig}_partial_r2: shape={r2.shape}, mean={np.nanmean(r2):.4f}, max={np.nanmax(r2):.4f}")

In [ ]:
# Reconstruct the figure inputs from save_dict (identical whether loaded or freshly computed).
age_grid    = save_dict['age_grid']
sa_axis_reg = save_dict['sa_axis_reg']
n_deciles   = save_dict['n_deciles']
r_ci_lo     = save_dict['r_ci_lo']
r_ci_hi     = save_dict['r_ci_hi']

results_reg = {sig: {'deriv': save_dict[f'{sig}_deriv'],     'pred': save_dict[f'{sig}_pred']}
               for sig in ('rest', 'task', 'deflection')}
results_dec = {sig: {'deriv': save_dict[f'{sig}_deriv_dec'], 'pred': save_dict[f'{sig}_pred_dec']}
               for sig in ('rest', 'task', 'deflection')}

cmap          = create_sydnor_sa_colormap()
series_colors = get_series_colors('decile', n_deciles, cmap)
draw_order    = get_draw_order('decile', n_deciles)

# Figure 4

In [ ]:
# --- Parameters ---
signal_type = 'deflection'  # 'rest', 'task', or 'deflection'
plot_level  = 'region'      # 'region' or 'decile'
# ------------------

_signal_labels = {'rest': 'REST INTs', 'task': 'Task INTs', 'deflection': 'INT Deflections'}
lw = 1.8

# Heatmap data
_reg_deriv = results_reg[signal_type]['deriv']

if plot_level == 'region':
    sort_idx     = np.argsort(sa_axis_reg)
    deriv_sorted = _reg_deriv[sort_idx]
    ylabel       = 'S-A axis'
    bottom_label = 'Sensorimotor pole'
    top_label    = 'Association pole'
else:
    deriv_sorted = results_dec[signal_type]['deriv']
    ylabel       = 'S-A axis decile'
    bottom_label = 'Sensorimotor'
    top_label    = 'Association'

n_rows = deriv_sorted.shape[0]
vmax   = np.nanpercentile(np.abs(deriv_sorted), 98)

crossing_ages = []
for i in range(n_rows):
    row = deriv_sorted[i, :]
    for t in range(len(row) - 1):
        if row[t] < 0 and row[t + 1] >= 0:
            frac = -row[t] / (row[t + 1] - row[t])
            crossing_ages.append(age_grid[t] + frac * (age_grid[t + 1] - age_grid[t]))
            break
mean_cross_age = np.mean(crossing_ages) if crossing_ages else None

r_vals = compute_deriv_sa_corr(_reg_deriv, sa_axis_reg, method='pearson')

# Figure layout
fig      = plt.figure(figsize=(8.5, 5), dpi=300)
gs_outer = GridSpec(1, 2, figure=fig, width_ratios=[1, 1], wspace=0.15)

# Left column: heatmap | narrow vertical colorbar
gs_left   = GridSpecFromSubplotSpec(1, 2, subplot_spec=gs_outer[0],
                                    width_ratios=[1, 0.025], wspace=0.05)
ax_heat   = fig.add_subplot(gs_left[0])
ax_cbar_h = fig.add_subplot(gs_left[1])

# Right column: pred / deriv / Pearson r (3 panels; SA-decile colorbar added as overlay)
gs_right = GridSpecFromSubplotSpec(3, 1, subplot_spec=gs_outer[1],
                                   height_ratios=[1, 1, 1], hspace=0.3)
ax0 = fig.add_subplot(gs_right[0])
ax1 = fig.add_subplot(gs_right[1], sharex=ax0)
ax2 = fig.add_subplot(gs_right[2], sharex=ax0)

# Heatmap
im = ax_heat.imshow(
    deriv_sorted, aspect='auto', origin='lower',
    extent=[age_grid[0], age_grid[-1], 0, n_rows],
    cmap='berlin', vmin=-vmax, vmax=vmax, interpolation='nearest',
)
if mean_cross_age is not None:
    ax_heat.axvline(mean_cross_age, color='gray', linestyle='--', lw=0.75, alpha=0.8)

ax_heat.set_xlabel('Age (years)', fontsize=8, labelpad=2)
ax_heat.set_ylabel(ylabel, fontsize=8, labelpad=4)
if plot_level == 'region':
    ax_heat.set_yticks([0, n_rows // 2, n_rows])
    ax_heat.set_yticklabels(['', '', ''], fontsize=8)
else:
    ax_heat.set_yticks(np.arange(n_rows) + 0.5)
    ax_heat.set_yticklabels([str(i + 1) for i in range(n_rows)], fontsize=8)
ax_heat.tick_params(labelsize=8, pad=1)

ax_heat.annotate(top_label,    xy=(age_grid[0], n_rows), xytext=(-8, -1.5), rotation=90,
                 textcoords='offset points', fontsize=8, va='top',    annotation_clip=False)
ax_heat.annotate(bottom_label, xy=(age_grid[0], 0),      xytext=(-8,  1.5), rotation=90,
                 textcoords='offset points', fontsize=8, va='bottom', annotation_clip=False)
ax_heat.annotate('', xy=(age_grid[-1], n_rows + n_rows * 0.015),
                 xytext=(age_grid[0],  n_rows + n_rows * 0.015),
                 arrowprops=dict(arrowstyle='->', color='k', lw=0.8), annotation_clip=False)
ax_heat.text((age_grid[0] + age_grid[-1]) / 2, n_rows + n_rows * 0.025,
             'Age-resolved analysis', ha='center', va='bottom', fontsize=8, clip_on=False)

cb_h = fig.colorbar(im, cax=ax_cbar_h, orientation='vertical')
cb_h.set_label(f'{_signal_labels[signal_type]} (annualized change)', fontsize=8)
cb_h.ax.yaxis.set_label_position('right')
cb_h.ax.tick_params(labelsize=6)

# Line plots
for i in draw_order:
    ax0.plot(age_grid, results_dec[signal_type]['pred'][i], color=series_colors[i], lw=lw)
style_ax(ax0, 'Age (years)', f'{_signal_labels[signal_type]}', None, demean=True)
ax0.set_xlabel('')
ax0.set_box_aspect(0.8)
plt.setp(ax0.get_xticklabels(), visible=False)

for i in draw_order:
    ax1.plot(age_grid, results_dec[signal_type]['deriv'][i], color=series_colors[i], lw=lw)
style_ax(ax1, 'Age (years)', f'{_signal_labels[signal_type]}\n(derivatives)', None, demean=True)
ax1.set_xlabel('')
ax1.set_box_aspect(0.8)
plt.setp(ax1.get_xticklabels(), visible=False)

# r(SA axis × regional GAM derivative) with bootstrap 95% CI and annotations
plot_deriv_sa_corr_with_ci(ax2, age_grid, r_vals, r_ci_lo, r_ci_hi, lw=lw)
ax2.set_xlabel('Age (years)', fontsize=8, labelpad=1)
ax2.set_ylabel(f'Pearson $r$\n(derivative × S-A axis)', fontsize=8, labelpad=1)
ax2.tick_params(labelsize=8, pad=0.25)
ax2.set_box_aspect(0.8)
ax2.spines[['top', 'right']].set_visible(False)

# Nudge ax1 up so its decile colorbar clears ax2's annotations
fig.canvas.draw()
_p1 = ax1.get_position()
ax1.set_position([_p1.x0, _p1.y0 + 0.04, _p1.width, _p1.height])

# SA-decile colorbar positioned below ax1
fig.canvas.draw()
add_sa_decile_colorbar(fig, ax1, cmap, n_deciles)

fig.savefig(os.path.join(nas_scratch_dir, f'fig4_combined_{task_scan}.png'),
            dpi=1000, bbox_inches='tight', transparent=True)
plt.show()